# MVP de Engenharia de Dados — Etapas 3 e 4: Modelagem e Carga

**Tema:** ocorrências criminais registradas no município de Sorocaba (SP)

---

Este notebook executa o **componente de ETL** da arquitetura de business intelligence e carrega o **data warehouse dimensional**. Na definição da Aula 1:

> "Extração, transformação e carga é o componente da arquitetura do ambiente de BI que operacionaliza o processo de alimentação do DW a partir de fontes de dados operacionais. Diversas questões de integração (padronização, heterogeneidade semântica) e tratamento da qualidade dos dados (tratamento de dados faltantes, resolução de inconsistências) são tratadas durante a execução desse processo."

O trabalho é dividido entre duas ferramentas, cada uma no que faz melhor:

| Etapa | Onde roda | Por quê |
|---|---|---|
| Extração, conciliação de esquemas, limpeza e filtro | **PySpark no Dataproc Serverless** | são 5,3 milhões de registros do estado inteiro, e o filtro por município só pode ser aplicado depois de lê-los; é o cluster Hadoop-Spark da Aula 3, com o Cloud Storage no lugar do HDFS |
| Modelagem dimensional e carga do esquema estrela | **SQL no BigQuery** | é o SGBD que hospeda o DW; a construção das dimensões e do fato usa DDL e DML da disciplina de Banco de Dados |

Todo o SQL executado aqui está versionado em [`sql/`](../sql/) — o notebook apenas o submete e mostra o resultado.

## 0. Parâmetros, autenticação e código-fonte

In [ ]:
# --- Parâmetros do ambiente (os mesmos do notebook 01) -------------------
PROJETO_ID = "mvp-criminalidade-sorocaba"
REGIAO     = "southamerica-east1"
BUCKET     = f"{PROJETO_ID}-datalake"
CONTA_SERVICO = f"etl-criminalidade@{PROJETO_ID}.iam.gserviceaccount.com"

# Repositório público do trabalho, de onde vêm o job Spark e os scripts SQL
REPO_URL  = "https://github.com/SEU_USUARIO/mvp-engenharia-dados.git"
REPO_DIR  = "/content/mvp-engenharia-dados"

from google.colab import auth
auth.authenticate_user()
!gcloud config set project {PROJETO_ID} --quiet

from google.cloud import storage, bigquery
cliente_gcs = storage.Client(project=PROJETO_ID)
cliente_bq  = bigquery.Client(project=PROJETO_ID)
bucket      = cliente_gcs.bucket(BUCKET)
print("Projeto:", PROJETO_ID)

In [ ]:
# Traz o código versionado para dentro do ambiente de execução
import os
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

!ls {REPO_DIR}/sql {REPO_DIR}/spark

## 1. Execução do job Spark

O script [`spark/etl_ocorrencias.py`](../spark/etl_ocorrencias.py) é enviado ao Cloud Storage e submetido ao Dataproc Serverless. Ele executa, nesta ordem:

1. **Extração** — lê a zona preparada inteira (os cinco anos em Parquet), com `mergeSchema` para unir arquivos que têm conjuntos de colunas diferentes.
2. **Conciliação de esquemas** — aplica o de-para levantado no notebook 01, reduzindo as várias grafias de cada ano (`CIDADE` × `NOME_MUNICIPIO`, `CD_IBGE` × `COD IBGE`, `CIRCUNCRIÇÃO` × `CIRCUNSCRICAO`) a um único conjunto de campos.
3. **Limpeza** — converte as sentinelas textuais da fonte (`NULL`, `(Vazio)`, `-`) e a coordenada zero em nulo de verdade.
4. **Filtro** — mantém apenas o município de Sorocaba, pelo **código IBGE da circunscrição**. O levantamento do notebook 01 mostrou que esse código acompanha o local onde o fato ocorreu, e não o município onde o boletim foi registrado — que pode ser outro, no caso dos registros feitos pela Delegacia Eletrônica. Como o objetivo é medir a criminalidade *em* Sorocaba, a circunscrição é o critério correto.
5. **Tipagem e derivações** — datas, horas e coordenadas assumem seus tipos; o tipo de local é derivado do subtipo nos anos em que a fonte não o publicou; o período do dia é derivado da hora quando ausente. Cada derivação é marcada em uma coluna de procedência.
6. **Padronização** — a mesma natureza criminal aparece na fonte com acentuações e traços diferentes; sem padronizar, uma única natureza seria contada como duas.
7. **Carga** — grava a tabela conformada no BigQuery e uma cópia em Parquet no data lake.

In [ ]:
import datetime as dt

# Envia o job para o data lake, de onde o Dataproc o lê
caminho_job = f"gs://{BUCKET}/codigo/etl_ocorrencias.py"
bucket.blob("codigo/etl_ocorrencias.py").upload_from_filename(
    f"{REPO_DIR}/spark/etl_ocorrencias.py")
print("job enviado:", caminho_job)

nome_batch = f"etl-ocorrencias-{dt.datetime.now():%Y%m%d-%H%M%S}"
print("batch:", nome_batch)

In [ ]:
!gcloud dataproc batches submit pyspark {caminho_job} \
    --batch={nome_batch} \
    --region={REGIAO} \
    --version=2.2 \
    --service-account={CONTA_SERVICO} \
    --deps-bucket=gs://{BUCKET} \
    --subnet=default \
    -- --projeto={PROJETO_ID} --bucket={BUCKET}

In [ ]:
# Conferência do que o job produziu
consulta = f"""
SELECT
  ano_arquivo,
  COUNT(*)                                   AS registros,
  COUNT(DISTINCT CONCAT(num_bo, '/', CAST(ano_bo AS STRING))) AS boletins,
  MIN(data_ocorrencia)                       AS ocorrencia_mais_antiga,
  MAX(data_ocorrencia)                       AS ocorrencia_mais_recente
FROM `{PROJETO_ID}.stg.ocorrencias_sorocaba`
GROUP BY ano_arquivo
ORDER BY ano_arquivo
"""
cliente_bq.query(consulta).to_dataframe()

## 2. Carga dos dados de referência do IBGE

Os dois arquivos JSON coletados na zona bruta viram tabelas na área de staging. São eles que permitem, adiante, validar o município e calcular a taxa por 100 mil habitantes.

Sobre a série de população, o que a fonte oferece e o que falta:

| Ano | Situação |
|---|---|
| 2022 | Censo Demográfico (tabela 4709) |
| 2023 | **sem número oficial publicado** |
| 2024 | estimativa populacional (tabela 6579) |
| 2025 | estimativa populacional (tabela 6579) |
| 2026 | **sem número oficial publicado** |

O tratamento dessas lacunas é decidido no SQL de carga, e não aqui — e fica declarado na coluna `origem_populacao` de cada linha.

In [ ]:
import json
import pandas as pd

def ler_json_da_zona_bruta(nome):
    return json.loads(bucket.blob(f"bruta/ibge/{nome}").download_as_bytes())

# --- Município: dado de referência da API de Localidades -----------------
municipio = ler_json_da_zona_bruta("municipio_3552205.json")
df_municipio = pd.DataFrame([{
    "cod_ibge":             str(municipio["id"]),
    "nome_municipio":       municipio["nome"],
    "uf":                   municipio["microrregiao"]["mesorregiao"]["UF"]["sigla"],
    "regiao_imediata":      municipio["regiao-imediata"]["nome"],
    "regiao_intermediaria": municipio["regiao-imediata"]["regiao-intermediaria"]["nome"],
    "mesorregiao":          municipio["microrregiao"]["mesorregiao"]["nome"],
    "microrregiao":         municipio["microrregiao"]["nome"],
}])
display(df_municipio)

# --- População: estimativas (2024, 2025) e Censo (2022) ------------------
linhas = []
for registro in ler_json_da_zona_bruta("populacao_estimativas_t6579.json")[1:]:
    if registro["V"].isdigit() and int(registro["D2N"]) >= 2022:
        linhas.append({"ano": int(registro["D2N"]),
                       "populacao": int(registro["V"]),
                       "origem_populacao": "estimativa"})

censo = ler_json_da_zona_bruta("populacao_censo2022_t4709.json")[1:]
linhas.append({"ano": 2022,
               "populacao": int(censo[0]["V"]),
               "origem_populacao": "censo"})

df_populacao = pd.DataFrame(linhas).sort_values("ano").reset_index(drop=True)
display(df_populacao)

In [ ]:
def carregar_dataframe(df, tabela, descricao):
    destino = f"{PROJETO_ID}.stg.{tabela}"
    configuracao = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    cliente_bq.load_table_from_dataframe(df, destino, job_config=configuracao).result()
    tabela_bq = cliente_bq.get_table(destino)
    tabela_bq.description = descricao
    cliente_bq.update_table(tabela_bq, ["description"])
    print(f"{destino}: {tabela_bq.num_rows} linhas")

carregar_dataframe(df_municipio, "municipio_ibge",
                   "Dado de referência do município, coletado da API de Localidades do IBGE.")
carregar_dataframe(df_populacao, "populacao_ibge",
                   "População residente de Sorocaba por ano, do IBGE. Apenas anos com número oficial publicado.")

## 3. Criação do esquema estrela

Os scripts SQL são executados na ordem numérica. Cada um está comentado com a decisão de modelagem que implementa:

| Script | O que faz |
|---|---|
| [`10_ddl_dw.sql`](../sql/10_ddl_dw.sql) | cria as sete dimensões e os dois fatos, com chaves primárias e estrangeiras declaradas e a descrição de cada coluna gravada na própria plataforma |
| [`15_de_para_natureza.sql`](../sql/15_de_para_natureza.sql) | tabela de referência que classifica cada natureza criminal por título do Código Penal |
| [`20_carga_dimensoes.sql`](../sql/20_carga_dimensoes.sql) | carrega as dimensões, gerando as chaves surrogate |
| [`21_carga_fato.sql`](../sql/21_carga_fato.sql) | carrega o fato, trocando cada descritor pela chave da dimensão |
| [`25_carga_populacao.sql`](../sql/25_carga_populacao.sql) | carrega o fato de população e trata os anos sem número oficial |

Todos são idempotentes: reexecutá-los produz exatamente o mesmo resultado, sem duplicar linhas.

In [ ]:
def executar_arquivo_sql(nome_arquivo):
    """Lê o script versionado, substitui o projeto e executa no BigQuery."""
    caminho = f"{REPO_DIR}/sql/{nome_arquivo}"
    with open(caminho, encoding="utf-8") as arquivo:
        sql = arquivo.read().replace("@projeto", PROJETO_ID)

    inicio = dt.datetime.now()
    job = cliente_bq.query(sql)
    job.result()
    duracao = dt.datetime.now() - inicio
    processado = (job.total_bytes_processed or 0) / 1e6
    print(f"  {nome_arquivo}: concluído em {duracao} ({processado:.1f} MB processados)")


for arquivo in ["10_ddl_dw.sql",
                "15_de_para_natureza.sql",
                "20_carga_dimensoes.sql",
                "21_carga_fato.sql",
                "25_carga_populacao.sql"]:
    executar_arquivo_sql(arquivo)

## 4. Verificação da carga

Três conferências, antes de qualquer análise:

1. **Volume de cada tabela** — nenhuma dimensão pode estar vazia.
2. **Conservação da medida** — a soma da medida no fato tem de ser idêntica ao número de linhas conformadas pelo ETL; qualquer diferença significa registro perdido ou duplicado na junção.
3. **Integridade referencial** — como as chaves estrangeiras do BigQuery são declaradas `NOT ENFORCED`, o banco não as verifica sozinho; a verificação é feita explicitamente.

In [ ]:
consulta = f"""
SELECT 'dim_tempo'            AS tabela, COUNT(*) AS linhas FROM `{PROJETO_ID}.dw.dim_tempo`
UNION ALL SELECT 'dim_periodo_dia',      COUNT(*) FROM `{PROJETO_ID}.dw.dim_periodo_dia`
UNION ALL SELECT 'dim_natureza',         COUNT(*) FROM `{PROJETO_ID}.dw.dim_natureza`
UNION ALL SELECT 'dim_local',            COUNT(*) FROM `{PROJETO_ID}.dw.dim_local`
UNION ALL SELECT 'dim_bairro',           COUNT(*) FROM `{PROJETO_ID}.dw.dim_bairro`
UNION ALL SELECT 'dim_delegacia',        COUNT(*) FROM `{PROJETO_ID}.dw.dim_delegacia`
UNION ALL SELECT 'dim_area_pm',          COUNT(*) FROM `{PROJETO_ID}.dw.dim_area_pm`
UNION ALL SELECT 'dim_municipio',        COUNT(*) FROM `{PROJETO_ID}.dw.dim_municipio`
UNION ALL SELECT 'fato_ocorrencia',      COUNT(*) FROM `{PROJETO_ID}.dw.fato_ocorrencia`
UNION ALL SELECT 'fato_populacao_anual', COUNT(*) FROM `{PROJETO_ID}.dw.fato_populacao_anual`
ORDER BY tabela
"""
cliente_bq.query(consulta).to_dataframe()

In [ ]:
consulta = f"""
WITH conferencia AS (
  SELECT
    (SELECT COUNT(*)              FROM `{PROJETO_ID}.stg.ocorrencias_sorocaba`) AS linhas_conformadas,
    (SELECT SUM(qtd_ocorrencia)   FROM `{PROJETO_ID}.dw.fato_ocorrencia`)       AS medida_no_fato,
    (SELECT COUNT(*) FROM `{PROJETO_ID}.dw.fato_ocorrencia` WHERE sk_tempo_ocorrencia = -1) AS sem_data,
    (SELECT COUNT(*) FROM `{PROJETO_ID}.dw.fato_ocorrencia` WHERE sk_natureza        = -1) AS sem_natureza,
    (SELECT COUNT(*) FROM `{PROJETO_ID}.dw.fato_ocorrencia` WHERE sk_municipio       = -1) AS sem_municipio
)
SELECT *,
       IF(linhas_conformadas = medida_no_fato, 'OK', 'DIVERGENTE') AS conservacao_da_medida
FROM conferencia
"""
cliente_bq.query(consulta).to_dataframe().T

In [ ]:
# Amostra do data warehouse já modelado, com as dimensões resolvidas
consulta = f"""
SELECT
  t.data                AS data_ocorrencia,
  p.periodo,
  n.categoria,
  n.natureza_apurada,
  l.tipo_local,
  b.nome_bairro,
  d.delegacia,
  f.qtd_ocorrencia
FROM `{PROJETO_ID}.dw.fato_ocorrencia`      AS f
JOIN `{PROJETO_ID}.dw.dim_tempo`            AS t ON t.sk_tempo       = f.sk_tempo_ocorrencia
JOIN `{PROJETO_ID}.dw.dim_periodo_dia`      AS p ON p.sk_periodo_dia = f.sk_periodo_dia
JOIN `{PROJETO_ID}.dw.dim_natureza`         AS n ON n.sk_natureza    = f.sk_natureza
JOIN `{PROJETO_ID}.dw.dim_local`            AS l ON l.sk_local       = f.sk_local
JOIN `{PROJETO_ID}.dw.dim_bairro`           AS b ON b.sk_bairro      = f.sk_bairro
JOIN `{PROJETO_ID}.dw.dim_delegacia`        AS d ON d.sk_delegacia   = f.sk_delegacia
ORDER BY t.data DESC
LIMIT 10
"""
cliente_bq.query(consulta).to_dataframe()

---

## Encerramento das etapas de modelagem e carga

| Item | Resultado |
|---|---|
| ETL | job PySpark no Dataproc Serverless, lendo os 5,3 milhões de registros do estado e conformando os de Sorocaba |
| Esquema | estrela, com sete dimensões e dois fatos compartilhando a dimensão município |
| Metadados | descrição de cada tabela e de cada coluna gravada no próprio BigQuery |
| Integridade | medida conservada entre a staging e o fato; nenhuma chave estrangeira órfã |

**Próxima etapa:** [`03_qualidade_dados.ipynb`](03_qualidade_dados.ipynb) — análise da qualidade de cada atributo.